In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.interpolate as sciin
import sklearn as sk
from functools import reduce

# consistent per-source colors, reused across all comparison plots below
colors = {
    "EDC":        "#D6195E",
    "Elderfield": "#1E88E5",
    "LR04":       "#43A047",
    "Honisch":    "#FB8C00",
    "Chalk":      "#6519C5",
}

In [ ]:
# --- Fourier spectrum comparison: setup -----------------------------------
# Mg/Ca ages aren't strictly increasing (24 duplicate depths); dedupe the same
# way dO18/CO2 were handled above before building a PCHIP spline.
MgCa_elderfield = MgCa_elderfield.drop_duplicates(subset='Age[kaBP]').reset_index(drop=True)

mgcaspline      = sciin.PchipInterpolator(MgCa_elderfield['Age[kaBP]'], MgCa_elderfield['Mg/Ca[mmol/mol](Normalized)'])
dO18_efspline   = sciin.PchipInterpolator(dO18_elderfield['Age[kaBP]'], dO18_elderfield['δ18O_H2O[‰SMOW]'])
dO18_LR04spline = sciin.PchipInterpolator(dO18_LR04['Age[kaBP]'], dO18_LR04['d18O_stack[‰]'])
co2honspline    = sciin.PchipInterpolator(co2_Honisch['Age[kaBP]'], co2_Honisch['pCO2water_SST_wet[µatm](constantAlkalinity_Calculated)'])
co2chalkspline  = sciin.PchipInterpolator(co2_chalk['Age_kaBP'], co2_chalk['CO2_ppmv'])

fft_step_kyr = 1.0  # common resampling step (kyr) used for every spectrum comparison below

def resample(spline, native_scale, lo_kyr, hi_kyr, step_kyr=fft_step_kyr):
    """
    Evaluate `spline` on a uniform kyr-BP grid (required for FFT). `native_scale`
    converts kyr -> the spline's own native x-units: 1 for kaBP-native splines
    (dO18, dO18_ef, dO18_LR04, Mg/Ca, Honisch, Chalk), 1000 for yr-BP-native
    splines (ch4, co2 EDC).
    """
    t_kyr = np.arange(lo_kyr, hi_kyr, step_kyr)
    return t_kyr, spline(t_kyr * native_scale)

def plot_fourier_spectrum(series, timestep, label, ax, color=None, normalize=True, linestyle='-'):
    """
    Compute and plot the (log-log) Fourier amplitude spectrum of a uniformly
    sampled 1D series, same convention as the emulator-vs-SM90 spectrum plot in
    Train_Emulator_obs_SR-LPNE.ipynb. normalize=True z-scores the series first
    so proxies with different units/magnitudes are visually comparable.
    """
    series = np.asarray(series, dtype=float)
    series = (series - series.mean()) / series.std() if normalize else series - series.mean()

    n = len(series)
    coeff = np.abs(np.fft.fft(series))
    freq = np.fft.fftfreq(n=n, d=timestep)
    mask = freq > 0

    peak_freq = freq[mask][np.argmax(coeff[mask])]
    print(f"{label}: dominant period = {1/peak_freq:.1f} kyr (peak freq = {peak_freq:.5f} 1/kyr)")

    ax.plot(freq[mask], coeff[mask], label=label, color=color, alpha=0.8, linestyle=linestyle)
    return freq[mask], coeff[mask]

def mark_cycles(ax):
    ymin, ymax = ax.get_ylim()
    ax.vlines(1/100, ymin=ymin, ymax=ymax, alpha=0.5, linestyle="dashed", color="darkblue")
    ax.vlines(1/41,  ymin=ymin, ymax=ymax, alpha=0.5, linestyle="dashed", color="slateblue")

### Data download and setup

In [ ]:
ch4 = pd.read_csv("/Users/vivekveluvali/Desktop/KU_Classes/MPT_Emulator_Project/LPNE/Data/Observations/Florian_Data/edc-ch4-2008-noaa.txt", comment = "#", delim_whitespace = True)

dO18 = pd.read_csv("/Users/vivekveluvali/Desktop/KU_Classes/MPT_Emulator_Project/LPNE/Data/Observations/Florian_Data/EDC_d18O.tab", comment = "#", delim_whitespace = True)
dO18 = dO18.drop_duplicates(subset='Gas_age[kaBP]').reset_index(drop=True)
dO18_LR04 = pd.read_csv("/Users/vivekveluvali/Desktop/KU_Classes/MPT_Emulator_Project/LPNE/Data/Observations/Lisiecki_Raymo_2005/datasets/Global_stack_d18O.tab", comment = "#", delim_whitespace = True)
dO18_LR04 = dO18_LR04.drop_duplicates(subset='Age[kaBP]').reset_index(drop=True)

co2 = pd.read_csv("/Users/vivekveluvali/Desktop/KU_Classes/MPT_Emulator_Project/LPNE/Data/Observations/Florian_Data/co2-2008-composite-EDC-noaa_Vodstock.txt", comment = "#", delim_whitespace = True)
co2 = co2.drop_duplicates(subset='gas_ageBP').reset_index(drop=True)
co2_Honisch = pd.read_csv("/Users/vivekveluvali/Desktop/KU_Classes/MPT_Emulator_Project/LPNE/Data/Observations/Honisch2009_pCO2.tab", comment = "#", delim_whitespace = True)
co2_Honisch = co2_Honisch.drop_duplicates(subset='Age[kaBP]').reset_index(drop=True)
co2_Yamamoto = pd.read_csv("/Users/vivekveluvali/Desktop/KU_Classes/MPT_Emulator_Project/LPNE/Data/Observations/CO2_YamamotoEtAl2022.csv", comment = '#', delim_whitespace = True)
co2_Yamamoto = co2_Yamamoto.drop_duplicates(subset='Age').reset_index(drop=True)

insol = pd.read_csv('/Users/vivekveluvali/Desktop/KU_Classes/MPT_Emulator_Project/LPNE/Data/Observations/Florian_Data/huybers06_65north_labeled.csv')

dO18_elderfield = pd.read_csv('/Users/vivekveluvali/Desktop/KU_Classes/MPT_Emulator_Project/LPNE/Data/Observations/Elderfield_2012/datasets/181-1123_d18O.tab', comment = "#", delim_whitespace = True)
MgCa_elderfield = pd.read_csv('/Users/vivekveluvali/Desktop/KU_Classes/MPT_Emulator_Project/LPNE/Data/Observations/Elderfield_2012/datasets/181-1123_Mg_Ca.tab', comment = "#", delim_whitespace = True)
MgCa_elderfield = MgCa_elderfield.drop_duplicates(subset='Age[kaBP]').reset_index(drop=True)
print(ch4.head())
print(ch4['gas_age_calBP'].max()/10000, " ", ch4['gas_age_calBP'].min()/10000)
print(dO18.head())
print(dO18['Gas_age[kaBP]'].max()/10, " ", dO18['Gas_age[kaBP]'].min()/10)
print(co2.head())
print(co2['gas_ageBP'].max()/10000, " ", co2['gas_ageBP'].min()/10000)

#earliest usable date is 10.17452 10kyr

In [ ]:
datasets_path = "/Users/vivekveluvali/Desktop/KU_Classes/MPT_Emulator_Project/LPNE/Data/Observations/Chalk-etal_2017/datasets"

def load_chalk_co2(path, period_label):
    df = pd.read_csv(path, comment="#", sep="\t")
    # first two columns are Age [ka BP] and median CO2 [ppmv]; the rest are
    # probabilistic bound columns with duplicate/truncated header text
    df = df.rename(columns={df.columns[0]: "Age_kaBP", df.columns[1]: "CO2_ppmv"})
    df["period"] = period_label
    return df

co2_early = load_chalk_co2(f"{datasets_path}/165-999_early_MPT_CO2.tab", "early_MPT")
co2_late  = load_chalk_co2(f"{datasets_path}/165-999_late_Pleistocene_CO2.tab", "late_Pleistocene")

co2_chalk = pd.concat([co2_late, co2_early], ignore_index=True).sort_values("Age_kaBP").reset_index(drop=True)

co2_chalk


In [ ]:
print(MgCa_elderfield.head())

In [ ]:
fig, axs = plt.subplots(3,1,figsize=(10,10),sharex=False)
axs[0].hist(ch4['gas_age_calBP']/10000,bins=1000)
axs[0].set_xlabel("CH4 Gas Age [10 ka BP]")
axs[0].set_ylabel(f'ch4 {ch4["gas_age_calBP"].count()} samples')
axs[1].hist(dO18['Gas_age[kaBP]']/10,bins=1000)
axs[1].set_xlabel("d18O Gas Age [10 ka BP]")
axs[1].set_ylabel(f'dO18 {dO18["Gas_age[kaBP]"].count()} samples')
axs[2].hist(co2['gas_ageBP']/10000,bins=1000)
axs[2].set_xlabel("CO2 Gas Age [10 ka BP]")
axs[2].set_ylabel(f'co2 {co2["gas_ageBP"].count()} samples')

In [ ]:
co2_Yamamoto.head()

In [ ]:
fig, axs = plt.subplots(3,1,figsize=(10,10),sharex=False)
axs[0].hist(co2_Yamamoto['Age']/10,bins=1000)
axs[0].set_xlabel("CO2 (Yamamoto) Age [10 ka BP]")
axs[0].set_ylabel(f'co2_Yamamoto {co2_Yamamoto["Age"].count()} samples')
axs[1].hist(dO18_LR04['Age[kaBP]']/10,bins=1000)
axs[1].set_xlabel("d18O (LR04) Age [10 ka BP]")
axs[1].set_ylabel(f'dO18_LR04 {dO18_LR04["Age[kaBP]"].count()} samples')
axs[2].hist(MgCa_elderfield['Age[kaBP]']/10,bins=1000)
axs[2].set_xlabel("Mg/Ca (Elderfield) Age [10 ka BP]")
axs[2].set_ylabel(f'MgCa_elderfield {MgCa_elderfield["Age[kaBP]"].count()} samples')
plt.tight_layout()
plt.show()


Alternative: instead of binning sample counts (sensitive to bin width/edges and doesn't show *where* resolution locally degrades), plot the spacing between consecutive samples directly against age - a cleaner, bin-free measure of temporal resolution.


In [ ]:
fig, axs = plt.subplots(3,1,figsize=(12,10),sharex=False)

def plot_resolution(ax, df, age_col, label, color):
    age_sorted = df[age_col] 
    spacing = np.diff(age_sorted)  # kyr between consecutive samples
    ax.plot(age_sorted[1:], spacing, '.', markersize=3, color=color, alpha=0.6)
    ax.set_yscale('log')
    ax.set_xlabel("Age [ka BP]")
    ax.set_ylabel("Sample spacing [kyr] (log scale)")
    ax.set_title(f"{label}: mean spacing = {np.mean(spacing):.2f} kyr, n = {len(age_sorted)}")

plot_resolution(axs[0], co2_Yamamoto, 'Age', "CO2 (Yamamoto)", "tab:blue")
plot_resolution(axs[1], dO18_LR04, 'Age[kaBP]', "d18O (LR04)", "tab:orange")
plot_resolution(axs[2], MgCa_elderfield, 'Age[kaBP]', "Mg/Ca (Elderfield)", "tab:green")

plt.tight_layout()
plt.show()


### Plotting Data

In [ ]:
ch4_norm = (ch4['CH4_mean'] - ch4['CH4_mean'].mean()) / ch4['CH4_mean'].std()
ch4spline = sciin.PchipInterpolator(ch4['gas_age_calBP'], ch4_norm)
tch4_fine = np.linspace(ch4['gas_age_calBP'].min(),
                     ch4['gas_age_calBP'].max(),
                     len(ch4_norm))
ch4_fine = ch4spline(tch4_fine)

ch4elderspline = sciin.PchipInterpolator(MgCa_elderfield['Age[kaBP]'], MgCa_elderfield['Mg/Ca[mmol/mol](Normalized)'])
tch4_elder_fine = np.linspace(MgCa_elderfield['Age[kaBP]'].min(),
                        MgCa_elderfield['Age[kaBP]'].max(),
                        len(MgCa_elderfield['Mg/Ca[mmol/mol](Normalized)']))
ch4_elder_fine = ch4elderspline(tch4_elder_fine)

dO18spline = sciin.PchipInterpolator(dO18['Gas_age[kaBP]'], dO18['δ18O-O2[pmille]'])
tdO18_fine = np.linspace(dO18['Gas_age[kaBP]'].min(),
                     dO18['Gas_age[kaBP]'].max(),
                     len(dO18['δ18O-O2[pmille]']))
dO18_fine = dO18spline(tdO18_fine)

dO18elderspline = sciin.PchipInterpolator(dO18_elderfield['Age[kaBP]'], dO18_elderfield['δ18O_H2O[‰SMOW]'])
tdO18_elder_fine = np.linspace(dO18_elderfield['Age[kaBP]'].min(),
                        dO18_elderfield['Age[kaBP]'].max(),
                        len(dO18_elderfield['δ18O_H2O[‰SMOW]']))
dO18_elder_fine = dO18elderspline(tdO18_elder_fine)

dO18_LR04spline = sciin.PchipInterpolator(dO18_LR04['Age[kaBP]'][:1133], dO18_LR04['d18O_stack[‰]'][:1133])
tdO18_LR04_fine = np.linspace(dO18_LR04['Age[kaBP]'][:1133].min(),
                        dO18_LR04['Age[kaBP]'][:1133].max(),
                        len(dO18_LR04['d18O_stack[‰]'][:1133]))
dO18_LR04_fine = dO18_LR04spline(tdO18_LR04_fine)

co2spline = sciin.PchipInterpolator(co2['gas_ageBP'], co2['CO2'])
tco2_fine = np.linspace(co2['gas_ageBP'].min(),
                     co2['gas_ageBP'].max(),
                     len(co2['CO2']))
co2_fine = co2spline(tco2_fine)

co2honspline = sciin.PchipInterpolator(co2_Honisch['Age[kaBP]'], co2_Honisch['pCO2water_SST_wet[µatm](constantAlkalinity_Calculated)'])
tco2_honisch_fine = np.linspace(co2_Honisch['Age[kaBP]'].min(),
                             co2_Honisch['Age[kaBP]'].max(),
                             len(co2_Honisch['pCO2water_SST_wet[µatm](constantAlkalinity_Calculated)']))
co2_honisch_fine = co2honspline(tco2_honisch_fine)

fig, axs = plt.subplots(4,1,figsize=(10,10),sharex=True)
axs[0].scatter(ch4["gas_age_calBP"]/10000, ch4_norm, color = 'black', marker = 'x', alpha = 0.33, 
               label = 'ch4, EDC 2008')
axs[0].scatter(MgCa_elderfield['Age[kaBP]']/10, MgCa_elderfield['Mg/Ca[mmol/mol](Normalized)'], 
               color = '#1E88E5', marker = 'x', alpha = 0.33, label = 'Mg/Ca, Elderfield 2012')
#axs[0].plot(tch4_fine/10000, ch4_fine, 
#            color='#D6195E', alpha=0.8, label = 'ch4, EDC 2008 (interpolated)')
#axs[0].plot(tch4_elder_fine/10, ch4_elder_fine,
#            color='#B1A28D', alpha=0.8, label = 'Mg/Ca, Elderfield 2012 (interpolated)')
axs[0].set_ylabel(f'ch4 {ch4["gas_age_calBP"].count()} samples (Z scored)')
axs[0].legend()

axs[1].scatter(dO18['Gas_age[kaBP]']/10, dO18['δ18O-O2[pmille]'], 
               color = 'black', marker = 'x', alpha = 0.4, label = 'dO18, EDC')
axs[1].scatter(dO18_elderfield['Age[kaBP]']/10, dO18_elderfield['δ18O_H2O[‰SMOW]'], 
               color = '#1E88E5', marker = 'x', alpha = 0.4, label = 'dO18, Elderfield 2012')
axs[1].scatter(dO18_LR04['Age[kaBP]'][:1133]/10, dO18_LR04['d18O_stack[‰]'][:1133], 
               color = '#43A047', marker = 'x', alpha = 0.4, label = 'dO18, LR04')
#axs[1].plot(tdO18_fine/10, dO18_fine, color='#D6195E', alpha=0.8, label = 'dO18, EDC (interpolated)')
#axs[1].plot(tdO18_elder_fine/10, dO18_elder_fine, color='#B1A28D', alpha=0.8, label = 'dO18, Elderfield 2012 (interpolated)')
axs[1].set_ylabel(f'dO18 {dO18["Gas_age[kaBP]"].count()} samples')
axs[1].legend()

axs[2].scatter(co2['gas_ageBP']/10000, co2['CO2'], 
               color = 'black', marker = 'x', alpha = 0.1, label = 'co2, EDC')
axs[2].scatter(co2_Honisch['Age[kaBP]']/10, co2_Honisch['pCO2water_SST_wet[µatm](constantAlkalinity_Calculated)'], 
               color = '#1E88E5', marker = 'x', alpha = 0.7, label = 'co2, Honisch 2009')
axs[2].scatter(co2_Yamamoto['Age']/10, co2_Yamamoto['Est_CO2_linear'],
               color = '#D6195E', marker = 'x', alpha = 0.6, label = 'co2, Yamamoto 2022')
#axs[2].plot(tco2_fine/10000, co2_fine, 
#            color='#D6195E', alpha=0.5, label = 'co2, EDC (interpolated)')
#axs[2].plot(tco2_honisch_fine/10, co2_honisch_fine, 
#            color='black', alpha=0.3, label = 'co2, Honisch 2009 (interpolated)')
axs[2].scatter(co2_chalk['Age_kaBP']/10, co2_chalk['CO2_ppmv'], 
            color='#6519C5', marker = 'x', alpha=0.1, label = 'co2, Chalk 2017')
axs[2].set_xlabel("Gas Age [10ka BP]")
axs[2].set_ylabel(f'co2 {co2["gas_ageBP"].count()} samples')
axs[2].legend()

axs[3].plot(insol["Age[kaBP]"][0:1500]/10, insol['ISI_thresh0Wm2'][0:1500])



In [ ]:


fig, axs = plt.subplots(4,1,figsize=(10,10),sharex=True)
axs[0].scatter(MgCa_elderfield['Age[kaBP]']/10, MgCa_elderfield['Mg/Ca[mmol/mol](Normalized)'], 
               color = '#1E88E5', marker = 'x', alpha = 0.75, label = 'Mg/Ca, Elderfield 2012')
#axs[0].plot(tch4_fine/10000, ch4_fine, 
#            color='#D6195E', alpha=0.8, label = 'ch4, EDC 2008 (interpolated)')
#axs[0].plot(tch4_elder_fine/10, ch4_elder_fine,
#            color='#B1A28D', alpha=0.8, label = 'Mg/Ca, Elderfield 2012 (interpolated)')
axs[0].set_ylabel(f'ch4 {ch4["gas_age_calBP"].count()} samples (Z scored)')
axs[0].legend()

axs[1].scatter(dO18_LR04['Age[kaBP]'][:1133]/10, dO18_LR04['d18O_stack[‰]'][:1133], 
               color = '#43A047', marker = 'x', alpha = 0.75, label = 'dO18, LR04')
axs[1].invert_yaxis()
#axs[1].plot(tdO18_fine/10, dO18_fine, color='#D6195E', alpha=0.8, label = 'dO18, EDC (interpolated)')
#axs[1].plot(tdO18_elder_fine/10, dO18_elder_fine, color='#B1A28D', alpha=0.8, label = 'dO18, Elderfield 2012 (interpolated)')
axs[1].set_ylabel(f'dO18 {dO18["Gas_age[kaBP]"].count()} samples')
axs[1].legend()

axs[2].scatter(co2_Yamamoto['Age']/10, co2_Yamamoto['Est_CO2_linear'],
               color = '#D6195E', marker = 'x', alpha = 0.75, label = 'co2, Yamamoto 2022')
#axs[2].plot(tco2_fine/10000, co2_fine, 
#            color='#D6195E', alpha=0.5, label = 'co2, EDC (interpolated)')
#axs[2].plot(tco2_honisch_fine/10, co2_honisch_fine, 
#            color='black', alpha=0.3, label = 'co2, Honisch 2009 (interpolated)')
axs[2].set_xlabel("Gas Age [10ka BP]")
axs[2].set_ylabel(f'co2 {co2["gas_ageBP"].count()} samples')
axs[2].legend()

axs[3].plot(insol["Age[kaBP]"][0:1500]/10, insol['ISI_thresh0Wm2'][0:1500])



In [ ]:
print(MgCa_elderfield['Age[kaBP]'].min())
print(MgCa_elderfield['Age[kaBP]'].max())

print(dO18_LR04['Age[kaBP]'].min())
print(dO18_LR04['Age[kaBP]'].max())


print(co2_Yamamoto['Age'].max())
print(co2_Yamamoto['Age'].min())
#plt.plot(t, co2spline(t*1000))

### Interpolating Data with Gaussian Process Regression

In [ ]:
def fitGPRtodata(x, y, x_fine, variance=None):
    # Fit a Gaussian Process Regressor to the data
    # normalize_y=True: Matern/RBF have no independent amplitude term (k(x,x)=1
    # is fixed), so without this the kernel's implicit unit variance can badly
    # mismatch a dataset's raw scale (e.g. CO2 in ppm, O(100s)) and the noise
    # term ends up absorbing most of the real signal, flattening the fit.
    if variance is not None:
        variance = variance/np.var(y) #normalize variance data
        kernel = sk.gaussian_process.kernels.Matern(length_scale=50, length_scale_bounds=(1, 1000), nu = 2.5)\
        #+ sk.gaussian_process.kernels.WhiteKernel(noise_level = variance, noise_level_bounds=(variance*1e-1, variance*1e3))
        gpr = sk.gaussian_process.GaussianProcessRegressor(kernel=kernel, alpha = variance, n_restarts_optimizer=5, normalize_y=True)
    else:
        kernel = sk.gaussian_process.kernels.Matern(length_scale=50, length_scale_bounds=(1, 1000), nu = 2.5)\
        + sk.gaussian_process.kernels.WhiteKernel(noise_level_bounds=(1e-5, 1e3))
        gpr = sk.gaussian_process.GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=5, normalize_y=True)
    gpr.fit(x.reshape(-1, 1), y)
    
    # Predict on the fine grid
    y_fine, y_std = gpr.predict(x_fine.reshape(-1, 1), return_std=True)
    
    return y_fine, y_std

In [ ]:
# --- GPR regression comparison, one panel per variable type ----------------
# Reuses fitGPRtodata (defined above) and the shared tgrid (real kyr BP, from
# the CH4 fit above). Per source: if the raw file has a per-sample stddev/error
# column, it's converted to variance and fed into alpha; otherwise variance=None
# so fitGPRtodata falls back to a plain Matern + WhiteKernel fit.
tgrid = np.linspace(0, 1461, 1462)


# Panel 1: CH4 (EDC) vs Mg/Ca (Elderfield 2012)
# CH4 has a listed 1-sigma column (CH4_1s) -> converted into the same z-scored
# space as ch4_norm and fed as alpha. Mg/Ca has no listed error -> WhiteKernel only.
X_ch4 = ch4["gas_age_calBP"].to_numpy() / 1000
y_ch4 = ch4_norm.to_numpy()
var_ch4 = ((ch4["CH4_1s"] / ch4["CH4_mean"].std()) ** 2).to_numpy()
regressed_ch4, ch4_std = fitGPRtodata(X_ch4, y_ch4, tgrid, variance=var_ch4)

X_mgca = MgCa_elderfield["Age[kaBP]"].to_numpy()
y_mgca = MgCa_elderfield["Mg/Ca[mmol/mol](Normalized)"].to_numpy()
regressed_mgca, mgca_std = fitGPRtodata(X_mgca, y_mgca, tgrid, variance=None)


# Panel 2: dO18 - EDC, Elderfield 2012, LR04
# only LR04 has a listed stddev column (d18O_StdDev_pm) -> fed as alpha.
# EDC and Elderfield have no listed error -> WhiteKernel only.
X_dO18 = dO18["Gas_age[kaBP]"].to_numpy()
y_dO18 = dO18["δ18O-O2[pmille]"].to_numpy()
regressed_dO18, dO18_std = fitGPRtodata(X_dO18, y_dO18, tgrid, variance=None)

X_dO18_ef = dO18_elderfield["Age[kaBP]"].to_numpy()
y_dO18_ef = dO18_elderfield["δ18O_H2O[‰SMOW]"].to_numpy()
regressed_dO18_ef, dO18_ef_std = fitGPRtodata(X_dO18_ef, y_dO18_ef, tgrid, variance=None)

X_dO18_LR04 = dO18_LR04["Age[kaBP]"].to_numpy()
y_dO18_LR04 = dO18_LR04["d18O_stack[‰]"].to_numpy()
var_dO18_LR04 = (dO18_LR04["d18O_StdDev_pm"] ** 2).to_numpy()
regressed_dO18_LR04, dO18_LR04_std = fitGPRtodata(X_dO18_LR04, y_dO18_LR04, tgrid, variance=var_dO18_LR04)


# Panel 3: CO2 - EDC, Honisch 2009, Chalk 2017
# Honisch has a listed stddev column (pCO2StdDev) -> fed as alpha. EDC has no
# listed error. Chalk's uncertainty columns are messy per-row probability
# bounds rather than a clean stddev, so it also falls back to WhiteKernel only.
X_co2 = co2["gas_ageBP"].to_numpy() / 1000
y_co2 = co2["CO2"].to_numpy()
regressed_co2, co2_std = fitGPRtodata(X_co2, y_co2, tgrid, variance=None)

X_co2_hon = co2_Honisch["Age[kaBP]"].to_numpy()
y_co2_hon = co2_Honisch["pCO2water_SST_wet[µatm](constantAlkalinity_Calculated)"].to_numpy()
var_co2_hon = (co2_Honisch["pCO2StdDev[±](uncertainty_Calculated)"] ** 2).to_numpy()
regressed_co2_hon, co2_hon_std = fitGPRtodata(X_co2_hon, y_co2_hon, tgrid, variance=var_co2_hon)

X_co2_chalk = co2_chalk["Age_kaBP"].to_numpy()
y_co2_chalk = co2_chalk["CO2_ppmv"].to_numpy()
regressed_co2_chalk, co2_chalk_std = fitGPRtodata(X_co2_chalk, y_co2_chalk, tgrid, variance=None)

X_co2_yamamoto = co2_Yamamoto['Age'].to_numpy()
y_co2_yamamoto = co2_Yamamoto['Est_CO2_linear'].to_numpy()
regressed_co2_yamamoto, co2_yamamoto_std = fitGPRtodata(X_co2_yamamoto, y_co2_yamamoto, tgrid, variance=12)


In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(20, 15))
tgrid = np.linspace(0, 1461, 1462)

axs[0].plot(tgrid, regressed_ch4, color=colors["EDC"], label="CH4 (EDC, regressed)")
axs[0].fill_between(tgrid, regressed_ch4 - ch4_std, regressed_ch4 + ch4_std, color=colors["EDC"], alpha=0.2)
axs[0].plot(tgrid, regressed_mgca, color=colors["Elderfield"], label="Mg/Ca (Elderfield, regressed)")
axs[0].fill_between(tgrid, regressed_mgca - mgca_std, regressed_mgca + mgca_std, color=colors["Elderfield"], alpha=0.2)
axs[0].set_title("CH4 (EDC) vs Mg/Ca (Elderfield 2012)")
axs[0].set_ylabel("value (z-scored)")
axs[0].legend(loc="upper right")

axs[1].plot(tgrid, regressed_dO18, color=colors["EDC"], label="dO18 (EDC, regressed)")
axs[1].fill_between(tgrid, regressed_dO18 - dO18_std, regressed_dO18 + dO18_std, color=colors["EDC"], alpha=0.2)
axs[1].plot(tgrid, regressed_dO18_ef, color=colors["Elderfield"], label="dO18 (Elderfield, regressed)")
axs[1].fill_between(tgrid, regressed_dO18_ef - dO18_ef_std, regressed_dO18_ef + dO18_ef_std, color=colors["Elderfield"], alpha=0.2)
axs[1].plot(tgrid, regressed_dO18_LR04, color=colors["LR04"], label="dO18 (LR04, regressed)")
axs[1].fill_between(tgrid, regressed_dO18_LR04 - dO18_LR04_std, regressed_dO18_LR04 + dO18_LR04_std, color=colors["LR04"], alpha=0.2)
axs[1].invert_yaxis()
axs[1].set_title("dO18: EDC vs Elderfield 2012 vs LR04")
axs[1].set_ylabel("value")
axs[1].legend(loc="upper right")


axs[2].plot(tgrid, regressed_co2, color=colors["EDC"], label="CO2 (EDC, regressed)")
axs[2].fill_between(tgrid, regressed_co2 - co2_std, regressed_co2 + co2_std, color=colors["EDC"], alpha=0.2)
axs[2].plot(tgrid, regressed_co2_hon, color=colors["Honisch"], label="CO2 (Honisch, regressed)")
axs[2].fill_between(tgrid, regressed_co2_hon - co2_hon_std, regressed_co2_hon + co2_hon_std, color=colors["Honisch"], alpha=0.2)
axs[2].plot(tgrid, regressed_co2_chalk, color=colors["Chalk"], label="CO2 (Chalk, regressed)")
axs[2].fill_between(tgrid, regressed_co2_chalk - co2_chalk_std, regressed_co2_chalk + co2_chalk_std, color=colors["Chalk"], alpha=0.2)
axs[2].plot(tgrid, regressed_co2_yamamoto, color="#8DB1A7", label="CO2 (Yamamoto, regressed)")
axs[2].fill_between(tgrid, regressed_co2_yamamoto - co2_yamamoto_std, regressed_co2_yamamoto + co2_yamamoto_std, color="#8DB1A7", alpha=0.2)
axs[2].set_title("CO2: EDC vs Honisch 2009 vs Chalk 2017 vs Yamamoto 2022")
axs[2].set_xlabel("Age [kyr BP]")
axs[2].set_ylabel("value")
axs[2].legend(loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
full_test_trajectory = pd.DataFrame({
    'Age[kaBP]': tgrid,
    'dO18_LR04[pm]': regressed_dO18_LR04,
    'dO18_std': dO18_LR04_std,
    'CO2_Yamamoto[pm]': regressed_co2_yamamoto,
    'CO2_std': co2_yamamoto_std,
    'MgCa': regressed_mgca,
    'MgCa_std': mgca_std,
    'insol': insol['ISI_thresh0Wm2'][0:1462]
})

full_test_trajectory
full_test_trajectory.to_csv("/Users/vivekveluvali/Desktop/KU_Classes/MPT_Emulator_Project/LPNE/Data/InterpolatedObs/GPR_full_traj_2diff.csv")

### Export Data

In [ ]:
t = np.arange(7, 799, 1) #starting from 7ka because that's the latest early time 
                         # note that the EDC dO18 data only goes back to ~101ka, so need to start there if I want to use EDC data
MgCa_Eld_int = pd.DataFrame({'Age[kaBP]':t, 'Mg/Ca_Eld[mmol/mol]': ch4elderspline(t)})
dO18_LR04_int = pd.DataFrame({'Age[kaBP]':t, 'dO18_LR04[pm]': dO18_LR04spline(t)})

CH4_EDC_int = pd.DataFrame({'Age[kaBP]':t, 'CH4_EDC[ppb]': ch4spline(t*1000)})
dO18_EDC_int = pd.DataFrame({'Age[kaBP]':t[95:], 'dO18_EDC[pm]': dO18spline(t[95:])})
CO2_EDC_int = pd.DataFrame({'Age[kaBP]':t, 'CO2_EDC[ppm]': co2spline(t*1000)})

insol_data = pd.DataFrame({'Age[kaBP]':t, 'Insolation[W/m2]': insol['ISI_thresh0Wm2'][7:799]})


frames = [MgCa_Eld_int, dO18_LR04_int, CH4_EDC_int, CO2_EDC_int, dO18_EDC_int, insol_data]

train_data_interpolated = reduce(
    lambda left, right: pd.merge(left, right, on='Age[kaBP]', how='outer'),
    frames
).sort_values('Age[kaBP]').reset_index(drop=True)

train_data_interpolated
train_data_interpolated.to_csv('/Users/vivekveluvali/Desktop/KU_Classes/MPT_Emulator_Project/LPNE/Data/InterpolatedObs/int_training.csv', index=False)



In [ ]:
# --- GPR-based export: regress each variable onto the export time grid `t` --
# Reuses the same (X, y, variance) definitions from the GPR comparison panels
# above. dO18 (EDC) and CO2 (Chalk) are intentionally excluded per spec.
gpr_vars = {
    "CH4_EDC[ppb]":        (X_ch4,       y_ch4,       var_ch4),
    "Mg/Ca_Eld[mmol/mol]": (X_mgca,      y_mgca,      None),
    "dO18_Eld[pm]":        (X_dO18_ef,   y_dO18_ef,   None),
    "dO18_LR04[pm]":       (X_dO18_LR04, y_dO18_LR04, var_dO18_LR04),
    "CO2_EDC[ppm]":        (X_co2,       y_co2,       None),
    "CO2_Hon[ppm]":        (X_co2_hon,   y_co2_hon,   var_co2_hon),
}

gpr_frames = []
for label, (X, y, var) in gpr_vars.items():
    mean, std = fitGPRtodata(X, y, t, variance=var)
    gpr_frames.append(pd.DataFrame({
        "Age[kaBP]": t,
        label: mean,
        f"{label}_std": std,
    }))

gpr_export_data = reduce(
    lambda left, right: pd.merge(left, right, on="Age[kaBP]", how="outer"),
    gpr_frames
).sort_values("Age[kaBP]").reset_index(drop=True)

#add insolation data to the gpr_export_data
gpr_export_data = pd.merge(gpr_export_data, insol_data, on="Age[kaBP]", how="outer").sort_values("Age[kaBP]").reset_index(drop=True)
gpr_export_data.head()

In [ ]:
gpr_export_data.to_csv('/Users/vivekveluvali/Desktop/KU_Classes/MPT_Emulator_Project/LPNE/Data/InterpolatedObs/gpr_training.csv', index=False)

In [ ]:
t_test = np.arange(808, 1009, 1) #starting from 7ka because that's the latest early time 
                         # note that the EDC dO18 data only goes back to ~101ka, so need to start there if I want to use EDC data
test_data = pd.DataFrame({'Age[kaBP]':t_test, 'Mg/Ca_Eld[mmol/mol]': ch4elderspline(t_test), 
                             'dO18_LR04[pm]': dO18_LR04spline(t_test), 
                             'CO2_Hon[ppm]': co2honspline(t_test),
                             'Insolation[W/m2]': insol['ISI_thresh0Wm2'][808:1009]}) 
test_data
test_data.to_csv('/Users/vivekveluvali/Desktop/KU_Classes/MPT_Emulator_Project/LPNE/Data/InterpolatedObs/int_testing.csv', index=False)


In [ ]:
gpr_vars2 = gpr_vars = {
    "Mg/Ca_Eld[mmol/mol]": (X_mgca,      y_mgca,      None),
    "dO18_Eld[pm]":        (X_dO18_ef,   y_dO18_ef,   None),
    "dO18_LR04[pm]":       (X_dO18_LR04, y_dO18_LR04, var_dO18_LR04),
    "CO2_Hon[ppm]":        (X_co2_hon,   y_co2_hon,   var_co2_hon),
}

gpr_frames2 = []
for label, (X, y, var) in gpr_vars2.items():
    mean, std = fitGPRtodata(X, y, t_test, variance=var)
    gpr_frames2.append(pd.DataFrame({
        "Age[kaBP]": t_test,
        label: mean,
        f"{label}_std": std,
    }))

gpr_export_data2 = reduce(
    lambda left, right: pd.merge(left, right, on="Age[kaBP]", how="outer"),
    gpr_frames2
).sort_values("Age[kaBP]").reset_index(drop=True)

insol_test = pd.DataFrame({
    "Age[kaBP]": t_test,
    "Insolation[W/m2]": insol['ISI_thresh0Wm2'][808:1009].to_numpy(),
})

gpr_export_data2 = pd.merge(gpr_export_data2, insol_test, on="Age[kaBP]", how="outer").sort_values("Age[kaBP]").reset_index(drop=True)
gpr_export_data2.head()
gpr_export_data2.to_csv('/Users/vivekveluvali/Desktop/KU_Classes/MPT_Emulator_Project/LPNE/Data/InterpolatedObs/gpr_testing.csv', index=False)

In [ ]:
t_test3 = np.arange(1087, 1243, 1)
gpr_vars3 = gpr_vars = {
    "Mg/Ca_Eld[mmol/mol]": (X_mgca,      y_mgca,      None),
    "dO18_Eld[pm]":        (X_dO18_ef,   y_dO18_ef,   None),
    "dO18_LR04[pm]":       (X_dO18_LR04, y_dO18_LR04, var_dO18_LR04),
    "CO2_Chalk[ppm]":        (X_co2_chalk,   y_co2_chalk,   None)
}

gpr_frames3 = []
for label, (X, y, var) in gpr_vars3.items():
    mean, std = fitGPRtodata(X, y, t_test3, variance=var)
    gpr_frames3.append(pd.DataFrame({
        "Age[kaBP]": t_test3,
        label: mean,
        f"{label}_std": std,
    }))

gpr_export_data3 = reduce(
    lambda left, right: pd.merge(left, right, on="Age[kaBP]", how="outer"),
    gpr_frames3
).sort_values("Age[kaBP]").reset_index(drop=True)

insol_test = pd.DataFrame({
    "Age[kaBP]": t_test3,
    "Insolation[W/m2]": insol['ISI_thresh0Wm2'][1087:1243].to_numpy(),
})

gpr_export_data3 = pd.merge(gpr_export_data3, insol_test, on="Age[kaBP]", how="outer").sort_values("Age[kaBP]").reset_index(drop=True)
gpr_export_data3.head()
gpr_export_data3.to_csv('/Users/vivekveluvali/Desktop/KU_Classes/MPT_Emulator_Project/LPNE/Data/InterpolatedObs/gpr_testing_long.csv', index=False)

### Time matching between LR04 dO18 and EDC dO18

In [ ]:
plt.scatter(dO18['Gas_age[kaBP]'][300:1100]/10, dO18['δ18O-O2[pmille]'][300:1100], 
               color = 'black', marker = 'x', alpha = 0.4, label = 'dO18, EDC')
plt.scatter(dO18_LR04['Age[kaBP]'][200:500]/10, dO18_LR04['d18O_stack[‰]'][200:500]*-1+3.5, 
               color = 'red', marker = 'x', alpha = 0.4, label = 'dO18, LR04')
plt.vlines(x = 24.3, ymin = -1, ymax = 2, color = 'black', alpha = 0.5, label = 'termination III?')
plt.vlines(x = 33.7, ymin = -1, ymax = 2, color = 'black', alpha = 0.5, label = 'termination III?')
plt.vlines(x = 42.4, ymin = -1, ymax = 2, color = 'black', alpha = 0.5, label = 'termination III?')
plt.legend()


In [ ]:
plt.scatter(dO18['Gas_age[kaBP]'][200:1400]/10, dO18['δ18O-O2[pmille]'][200:1400], 
               color = 'black', marker = 'x', alpha = 0.4, label = 'dO18, EDC')
plt.scatter(dO18_LR04['Age[kaBP]'][200:700]/10, dO18_LR04['d18O_stack[‰]'][200:700], 
               color = 'red', marker = 'x', alpha = 0.4, label = 'dO18, LR04')
plt.scatter(dO18_elderfield['Age[kaBP]'][200:700]/10, dO18_elderfield['δ18O_H2O[‰SMOW]'][200:700], 
               color = '#1E88E5', marker = 'x', alpha = 0.4, label = 'dO18, Elderfield 2012')

plt.vlines(x = 52.3, ymin = -1, ymax = 5, color = 'black', alpha = 0.5, label = 'termination VI?')
plt.vlines(x = 62.1, ymin = -1, ymax = 5, color = 'black', alpha = 0.5, label = 'termination VII?')
plt.vlines(x = 24.3, ymin = -1, ymax = 5, color = 'black', alpha = 0.5, label = 'termination VI?')
plt.vlines(x = 33.7, ymin = -1, ymax = 5, color = 'black', alpha = 0.5, label = 'termination VII?')
plt.vlines(x = 42.4, ymin = -1, ymax = 5, color = 'black', alpha = 0.5, label = 'termination VI?')
#plt.legend()

### Code written by Claude, need to take a closer look later

In [ ]:
# creating the splined observational data
# common time grid in "10 kyr" units, per your spec
t_10kyr = np.round(np.arange(10.2, 80.0 + 1e-9, 0.1), 1)
age_kyr = t_10kyr * 10  # same grid expressed in kyr BP, used for lookup

# native x-axis units differ per dataset (see the /10000, /10, /10000 factors
# used earlier to normalize each series to "10 ka" units for plotting):
#   ch4, co2 gas ages -> years BP   => age_kyr * 1000
#   dO18 gas age, insol age         => already ka BP / kyr BP => age_kyr directly

ch4_grid  = ch4spline(age_kyr * 1000)
dO18_grid = dO18spline(age_kyr)
co2_grid  = co2spline(age_kyr * 1000)

insolspline = sciin.PchipInterpolator(insol['age_kyrBP'], insol['ISI_thresh0Wm2'])
insol_grid  = insolspline(age_kyr)

obs_interp = pd.DataFrame({
    "t_10kyr":    t_10kyr,
    "age_kyrBP":  age_kyr,
    "CH4":        ch4_grid,
    "dO18":       dO18_grid,
    "CO2":        co2_grid,
    "Insolation": insol_grid,
})

obs_interp
fig, axs = plt.subplots(4,1,figsize=(10,10),sharex=True)
axs[0].plot(obs_interp['t_10kyr'], obs_interp['CH4'], color='#D6195E', alpha=0.8)
axs[0].set_ylabel(f'ch4 {ch4["gas_age_calBP"].count()} samples')
axs[1].plot(obs_interp['t_10kyr'], obs_interp['dO18'], color='#D6195E', alpha=0.8)
axs[1].set_ylabel(f'dO18 {dO18["Gas_age[kaBP]"].count()} samples')
axs[2].plot(obs_interp['t_10kyr'], obs_interp['CO2'], color='#D6195E', alpha=0.8)
axs[2].set_xlabel("Gas Age [10ka BP]")
axs[2].set_ylabel(f'co2 {co2["gas_ageBP"].count()} samples')
axs[3].plot(obs_interp['t_10kyr'], obs_interp['Insolation'][0:800])

obs_interp.to_csv('/Users/vivekveluvali/Desktop/KU_Classes/MPT_Emulator_Project/LPNE/Data/Observations/interpolateddata.csv')


In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(20, 20))

# Pane 1: CH4 (EDC) vs Mg/Ca (Elderfield 2012), resampled onto their overlapping age range
lo = max(ch4['gas_age_calBP'].min()/1000, MgCa_elderfield['Age[kaBP]'].min())
hi = min(ch4['gas_age_calBP'].max()/1000, MgCa_elderfield['Age[kaBP]'].max())
_, ch4_cmp  = resample(ch4spline, 1000, lo, hi)
_, mgca_cmp = resample(mgcaspline, 1,    lo, hi)
plot_fourier_spectrum(ch4_cmp,  fft_step_kyr, "CH4 (EDC)",          axs[0], color=colors["EDC"])
plot_fourier_spectrum(mgca_cmp, fft_step_kyr, "Mg/Ca (Elderfield)", axs[0], color=colors["Elderfield"])
axs[0].set_title("CH4 (EDC) vs Mg/Ca (Elderfield 2012)")

# Pane 2: dO18 from EDC, Elderfield 2012, and LR04, resampled onto their common overlap
lo = max(dO18['Gas_age[kaBP]'].min(), dO18_elderfield['Age[kaBP]'].min(), dO18_LR04['Age[kaBP]'].min())
hi = min(dO18['Gas_age[kaBP]'].max(), dO18_elderfield['Age[kaBP]'].max(), dO18_LR04['Age[kaBP]'].max())
_, dO18_cmp     = resample(dO18spline,      1, lo, hi)
_, dO18ef_cmp   = resample(dO18_efspline,   1, lo, hi)
_, dO18lr04_cmp = resample(dO18_LR04spline, 1, lo, hi)
plot_fourier_spectrum(dO18_cmp,     fft_step_kyr, "dO18 (EDC)",        axs[1], color=colors["EDC"])
plot_fourier_spectrum(dO18ef_cmp,   fft_step_kyr, "dO18 (Elderfield)", axs[1], color=colors["Elderfield"])
plot_fourier_spectrum(dO18lr04_cmp, fft_step_kyr, "dO18 (LR04)",       axs[1], color=colors["LR04"])
axs[1].set_title("dO18: EDC vs Elderfield 2012 vs LR04")

# Pane 3: CO2 from EDC and Chalk 2017 (Honisch dropped - too sparse/gappy to be
# a reliable spectrum, see prior note). EDC is resampled only over its own real
# data range (no extrapolation past where the ice-core record actually ends).
# Chalk is split into its two disjoint windows rather than forced onto one
# shared span, since the record itself has a ~830 kyr gap between them.
co2_lo, co2_hi = co2['gas_ageBP'].min()/1000, co2['gas_ageBP'].max()/1000
_, co2_cmp = resample(co2spline, 1000, co2_lo, co2_hi)
plot_fourier_spectrum(co2_cmp, fft_step_kyr, "CO2 (EDC)", axs[2], color=colors["EDC"])

chalk_late  = co2_chalk.loc[co2_chalk['period'] == 'late_Pleistocene', 'Age_kaBP']
chalk_early = co2_chalk.loc[co2_chalk['period'] == 'early_MPT', 'Age_kaBP']

chalk_late_lo,  chalk_late_hi  = max(chalk_late.min(),  0),    min(chalk_late.max(),  200)
chalk_early_lo, chalk_early_hi = max(chalk_early.min(), 1000), min(chalk_early.max(), 1200)

_, co2chalk_late_cmp  = resample(co2chalkspline, 1, chalk_late_lo,  chalk_late_hi)
_, co2chalk_early_cmp = resample(co2chalkspline, 1, chalk_early_lo, chalk_early_hi)

plot_fourier_spectrum(co2chalk_late_cmp,  fft_step_kyr, "CO2 (Chalk, 0-200 ka)",
                       axs[2], color=colors["Chalk"], linestyle="-")
plot_fourier_spectrum(co2chalk_early_cmp, fft_step_kyr, "CO2 (Chalk, 1.0-1.2 Mya)",
                       axs[2], color=colors["Chalk"], linestyle="--")
axs[2].set_title("CO2: EDC vs Chalk 2017 (0-200 ka & 1.0-1.2 Mya)")

for ax in axs:
    mark_cycles(ax)
    ax.set_yscale("log")
    ax.set_xscale("log")
    ax.set_xlabel("frequency [1/kyr]")
    ax.legend(loc="upper right", fontsize=9)
axs[0].set_ylabel("normalized coefficients")

fig.suptitle("dashed lines: 100 kyr (navy) and 41 kyr (slate) Milankovitch cycles")
plt.tight_layout()
plt.show()